In [ ]:
#official names but i dont wanna change them in the cells below
# spring_mean   = np.array([107.999996, 107.860557, 96.102010, 90.020693, 92.085084, 79.176802])
# spring_median = np.array([35.435, 34.820, 35.560, 35.620, 36.280, 35.900])
# spring_n      = np.array([32480, 37929, 49827, 54004, 79427, 108453])

# autumn_mean   = np.array([81.221291, 83.520147, 76.577716, 72.022322, 72.707952, 66.418481])
# autumn_median = np.array([36.94, 37.79, 37.25, 36.11, 36.72, 35.85])
# autumn_n      = np.array([28881, 34098, 44130, 51750, 69649, 89104])

In [ ]:
import numpy as np
from scipy import stats

years = np.array([2017, 2018, 2019, 2020, 2021, 2022])


# Data
years = np.array([2017, 2018, 2019, 2020, 2021, 2022])

# for spring
native_mean   = np.array([107.999996, 107.860557, 96.102010, 90.020693, 92.085084, 79.176802])
native_n      = np.array([32480, 37929, 49827, 54004, 79427, 108453])

# for autumn
migrating_mean = np.array([81.221291, 83.520147, 76.577716, 72.022322, 72.707952, 66.418481])
migrating_n    = np.array([28881, 34098, 44130, 51750, 69649, 89104])

# ─────────────────────────────────────────────
# 1. WEIGHTED LINEAR REGRESSION
# ─────────────────────────────────────────────

def weighted_regression(years, means, weights):
    """
    Weighted OLS: minimises sum of w_i * (y_i - (a + b*x_i))^2
    Returns slope, intercept, R², and p-value for the slope.
    """
    w = weights / weights.sum()           # normalise weights
    x = years - years.mean()              # centre year to reduce collinearity

    x_bar = np.average(x, weights=w)
    y_bar = np.average(means, weights=w)

    Sxx = np.sum(w * (x - x_bar)**2)
    Sxy = np.sum(w * (x - x_bar) * (means - y_bar))
    Syy = np.sum(w * (means - y_bar)**2)

    slope     = Sxy / Sxx
    intercept = y_bar - slope * x_bar

    fitted    = intercept + slope * x
    residuals = means - fitted
    SS_res    = np.sum(w * residuals**2)
    SS_tot    = Syy
    r2        = 1 - SS_res / SS_tot

    # Standard error of slope; using n-2 degrees of freedom
    n      = len(years)
    s2     = SS_res / (n - 2)
    se_b   = np.sqrt(s2 / Sxx)
    t_stat = slope / se_b
    p_val  = 2 * stats.t.sf(abs(t_stat), df=n - 2)

    return slope, intercept, r2, t_stat, p_val, se_b


native_slope, native_intercept, native_r2, native_t, native_p, native_se = \
    weighted_regression(years, native_mean, native_n)

migr_slope, migr_intercept, migr_r2, migr_t, migr_p, migr_se = \
    weighted_regression(years, migrating_mean, migrating_n)

print("=" * 55)
print("WEIGHTED LINEAR REGRESSION")
print("=" * 55)
for label, slope, intercept, r2, t, p, se in [
    ("Spring",    native_slope,  native_intercept,  native_r2,  native_t,  native_p,  native_se),
    ("Autumn",   migr_slope,    migr_intercept,    migr_r2,    migr_t,    migr_p,    migr_se),
]:
    print(f"\n{label}")
    print(f"  Slope       : {slope:+.4f} km/year  (SE = {se:.4f})")
    print(f"  Intercept   : {intercept:.4f}")
    print(f"  R²          : {r2:.4f}")
    print(f"  t-statistic : {t:.4f}")
    print(f"  p-value     : {p:.4f}  {'*** significant' if p < 0.05 else 'not significant'}")

# Interaction test: are the two slopes significantly different?
diff        = native_slope - migr_slope
se_diff     = np.sqrt(native_se**2 + migr_se**2)
t_diff      = diff / se_diff
p_diff      = 2 * stats.t.sf(abs(t_diff), df=(len(years) - 2) * 2)

print(f"\nSlope difference (spring − autumn): {diff:+.4f} km/year")
print(f"  SE of difference : {se_diff:.4f}")
print(f"  t-statistic      : {t_diff:.4f}")
print(f"  p-value          : {p_diff:.4f}  {'*** significant' if p_diff < 0.05 else 'not significant'}")

# ─────────────────────────────────────────────
# 2. MANN-KENDALL TREND TEST
# ─────────────────────────────────────────────

def mann_kendall(x):
    """
    Manual Mann-Kendall test.
    S  = sum of sign(x_j - x_i) for all i < j
    Var(S) uses the standard formula (no tied correction needed for continuous data).
    Returns S, tau, z-statistic, and p-value.
    """
    n = len(x)
    s = 0
    for i in range(n - 1):
        for j in range(i + 1, n):
            s += np.sign(x[j] - x[i])

    # Variance of S under H0
    var_s = n * (n - 1) * (2 * n + 5) / 18

    # Continuity-corrected z
    if s > 0:
        z = (s - 1) / np.sqrt(var_s)
    elif s < 0:
        z = (s + 1) / np.sqrt(var_s)
    else:
        z = 0.0

    p = 2 * stats.norm.sf(abs(z))

    # Kendall's tau
    tau = s / (n * (n - 1) / 2)

    return s, tau, z, p


native_s, native_tau, native_z, native_mk_p = mann_kendall(native_mean)
migr_s,   migr_tau,   migr_z,   migr_mk_p  = mann_kendall(migrating_mean)

print("\n" + "=" * 55)
print("MANN-KENDALL TREND TEST")
print("=" * 55)
for label, s, tau, z, p in [
    ("Spring",    native_s, native_tau, native_z, native_mk_p),
    ("Autumn",   migr_s,   migr_tau,   migr_z,   migr_mk_p),
]:
    direction = "downward" if tau < 0 else "upward"
    print(f"\n{label}")
    print(f"  S statistic : {s}")
    print(f"  Kendall tau : {tau:+.4f}  ({direction} trend)")
    print(f"  z-statistic : {z:.4f}")
    print(f"  p-value     : {p:.4f}  {'*** significant' if p < 0.05 else 'not significant'}")

WEIGHTED LINEAR REGRESSION

Native
  Slope       : -5.7521 km/year  (SE = 0.9826)
  Intercept   : 95.5827
  R²          : 0.8955
  t-statistic : -5.8539
  p-value     : 0.0042  *** significant

Migrating
  Slope       : -3.2547 km/year  (SE = 0.5599)
  Intercept   : 75.4327
  R²          : 0.8942
  t-statistic : -5.8130
  p-value     : 0.0044  *** significant

Slope difference (native − migrating): -2.4975 km/year
  SE of difference : 1.1309
  t-statistic      : -2.2083
  p-value          : 0.0582  not significant

MANN-KENDALL TREND TEST

Spring
  S statistic : -13.0
  Kendall tau : -0.8667  (downward trend)
  z-statistic : -2.2544
  p-value     : 0.0242  *** significant

Autumn
  S statistic : -11.0
  Kendall tau : -0.7333  (downward trend)
  z-statistic : -1.8787
  p-value     : 0.0603  not significant


#### i have no idea why we're doing this

In [3]:
from itertools import combinations

def mann_kendall_exact(x):
    """
    Exact Mann-Kendall p-value via full permutation of all n! orderings.
    Only feasible for small n (<=10 or so).
    """
    from itertools import permutations

    n = len(x)

    def compute_s(seq):
        s = 0
        for i in range(len(seq) - 1):
            for j in range(i + 1, len(seq)):
                s += np.sign(seq[j] - seq[i])
        return s

    observed_s = compute_s(x)

    # Generate all permutations and compute S for each
    all_s = [compute_s(perm) for perm in permutations(x)]
    total = len(all_s)

    # Two-tailed: count how often |S_perm| >= |observed_s|
    p_exact = sum(1 for s in all_s if abs(s) >= abs(observed_s)) / total

    tau = observed_s / (n * (n - 1) / 2)

    return observed_s, tau, p_exact


print("\n" + "=" * 55)
print("MANN-KENDALL — EXACT PERMUTATION P-VALUES")
print("=" * 55)
for label, data in [("Native", native_mean), ("Migrating", migrating_mean)]:
    s, tau, p = mann_kendall_exact(data)
    direction = "downward" if tau < 0 else "upward"
    print(f"\n{label}")
    print(f"  S statistic : {s}")
    print(f"  Kendall tau : {tau:+.4f}  ({direction} trend)")
    print(f"  p-value     : {p:.4f}  {'*** significant' if p < 0.05 else 'not significant'}")


MANN-KENDALL — EXACT PERMUTATION P-VALUES

Native
  S statistic : -9.0
  Kendall tau : -0.6000  (downward trend)
  p-value     : 0.1361  not significant

Migrating
  S statistic : -13.0
  Kendall tau : -0.8667  (downward trend)
  p-value     : 0.0167  *** significant


interpreting the results from the first cell: Weighted linear regression revealed a statistically significant negative trend in mean distance to the nearest city for both native birds (slope = −1.14 km/year, p = 0.029) and migrating birds (slope = −3.80 km/year, p = 0.006) over the 2017–2022 period, indicating that both groups are being observed increasingly closer to urban areas over time. A Mann-Kendall trend test corroborated the migrating birds result (τ = −0.87, p = 0.017), but did not reach significance for native birds (τ = −0.60, p = 0.136), suggesting that while the overall linear direction is downward, the native bird trend is not strictly monotonic across years. Critically, the difference in slopes between the two groups was itself statistically significant (p = 0.009), indicating that migrating birds are approaching urban areas at a rate approximately three times faster than native birds.

## robustness checks

In [ ]:
#official names but i dont wanna change them in the cells below
# spring_mean   = np.array([107.999996, 107.860557, 96.102010, 90.020693, 92.085084, 79.176802])
# spring_median = np.array([35.435, 34.820, 35.560, 35.620, 36.280, 35.900])
# spring_n      = np.array([32480, 37929, 49827, 54004, 79427, 108453])

# autumn_mean   = np.array([81.221291, 83.520147, 76.577716, 72.022322, 72.707952, 66.418481])
# autumn_median = np.array([36.94, 37.79, 37.25, 36.11, 36.72, 35.85])
# autumn_n      = np.array([28881, 34098, 44130, 51750, 69649, 89104])

In [4]:
import numpy as np
from scipy import stats
from itertools import permutations

years = np.array([2017, 2018, 2019, 2020, 2021, 2022])

native_median   = np.array([35.435, 34.820, 35.560, 35.620, 36.280, 35.900])
native_n        = np.array([32480, 37929, 49827, 54004, 79427, 108453])

migrating_median = np.array([36.94, 37.79, 37.25, 36.11, 36.72, 35.85])
migrating_n      = np.array([28881, 34098, 44130, 51750, 69649, 89104])

# ─────────────────────────────────────────────
# 1. WEIGHTED LINEAR REGRESSION ON MEDIAN
# ─────────────────────────────────────────────

def weighted_regression(years, values, weights):
    w = weights / weights.sum()
    x = years - years.mean()

    x_bar = np.average(x, weights=w)
    y_bar = np.average(values, weights=w)

    Sxx = np.sum(w * (x - x_bar)**2)
    Sxy = np.sum(w * (x - x_bar) * (values - y_bar))
    Syy = np.sum(w * (values - y_bar)**2)

    slope     = Sxy / Sxx
    intercept = y_bar - slope * x_bar

    fitted    = intercept + slope * x
    residuals = values - fitted
    SS_res    = np.sum(w * residuals**2)
    r2        = 1 - SS_res / Syy

    n      = len(years)
    s2     = SS_res / (n - 2)
    se_b   = np.sqrt(s2 / Sxx)
    t_stat = slope / se_b
    p_val  = 2 * stats.t.sf(abs(t_stat), df=n - 2)

    return slope, intercept, r2, t_stat, p_val, se_b


native_slope, native_intercept, native_r2, native_t, native_p, native_se = \
    weighted_regression(years, native_median, native_n)

migr_slope, migr_intercept, migr_r2, migr_t, migr_p, migr_se = \
    weighted_regression(years, migrating_median, migrating_n)

print("=" * 55)
print("WEIGHTED LINEAR REGRESSION ON MEDIAN")
print("=" * 55)
for label, slope, intercept, r2, t, p, se in [
    ("Spring",    native_slope, native_intercept, native_r2, native_t, native_p, native_se),
    ("Autumn",   migr_slope,    migr_intercept,    migr_r2,    migr_t,    migr_p,    migr_se),
]:
    print(f"\n{label}")
    print(f"  Slope       : {slope:+.4f} km/year  (SE = {se:.4f})")
    print(f"  Intercept   : {intercept:.4f}")
    print(f"  R²          : {r2:.4f}")
    print(f"  t-statistic : {t:.4f}")
    print(f"  p-value     : {p:.4f}  {'*** significant' if p < 0.05 else 'not significant'}")

diff    = native_slope - migr_slope
se_diff = np.sqrt(native_se**2 + migr_se**2)
t_diff  = diff / se_diff
p_diff  = 2 * stats.t.sf(abs(t_diff), df=(len(years) - 2) * 2)

print(f"\nSlope difference (spring − autumn): {diff:+.4f} km/year")
print(f"  SE of difference : {se_diff:.4f}")
print(f"  t-statistic      : {t_diff:.4f}")
print(f"  p-value          : {p_diff:.4f}  {'*** significant' if p_diff < 0.05 else 'not significant'}")

# ─────────────────────────────────────────────
# 2. MANN-KENDALL EXACT TEST ON MEDIAN
# ─────────────────────────────────────────────

def mann_kendall_exact(x):
    def compute_s(seq):
        s = 0
        for i in range(len(seq) - 1):
            for j in range(i + 1, len(seq)):
                s += np.sign(seq[j] - seq[i])
        return s

    observed_s = compute_s(x)
    all_s      = [compute_s(perm) for perm in permutations(x)]
    p_exact    = sum(1 for s in all_s if abs(s) >= abs(observed_s)) / len(all_s)
    tau        = observed_s / (len(x) * (len(x) - 1) / 2)

    return observed_s, tau, p_exact


print("\n" + "=" * 55)
print("MANN-KENDALL EXACT TEST ON MEDIAN")
print("=" * 55)
for label, data in [("spring", native_median), ("autumn", migrating_median)]:
    s, tau, p = mann_kendall_exact(data)
    direction = "downward" if tau < 0 else "upward"
    print(f"\n{label}")
    print(f"  S statistic : {s}")
    print(f"  Kendall tau : {tau:+.4f}  ({direction} trend)")
    print(f"  p-value     : {p:.4f}  {'*** significant' if p < 0.05 else 'not significant'}")

# ─────────────────────────────────────────────
# 3. COMPARISON SUMMARY: MEAN VS MEDIAN
# ─────────────────────────────────────────────

native_mean_vals   = np.array([41.865980, 45.332769, 44.491083, 41.682730, 40.224726, 40.813832])
migrating_mean_vals = np.array([90.610106, 89.488594, 81.937664, 77.234132, 79.641804, 70.922299])

n_slope_mean, _, _, _, n_p_mean, _ = weighted_regression(years, native_mean_vals, native_n)
m_slope_mean, _, _, _, m_p_mean, _ = weighted_regression(years, migrating_mean_vals, migrating_n)
_, _, n_mk_p_median = mann_kendall_exact(native_median)
_, _, m_mk_p_median = mann_kendall_exact(migrating_median)

# print("\n" + "=" * 55)
# print("COMPARISON SUMMARY: MEAN VS MEDIAN")
# print("=" * 55)
# print(f"\n{'':30s} {'Mean':>10s}  {'Median':>10s}")
# print(f"  {'Native — WLS slope (km/yr)':30s} {n_slope_mean:>+10.4f}  {native_slope:>+10.4f}")
# print(f"  {'Native — WLS p-value':30s} {n_p_mean:>10.4f}  {native_p:>10.4f}")
# print(f"  {'Native — MK p-value':30s} {n_mk_p_mean:>10.4f}  {n_mk_p_median:>10.4f}")
# print(f"  {'Migrating — WLS slope (km/yr)':30s} {m_slope_mean:>+10.4f}  {migr_slope:>+10.4f}")
# print(f"  {'Migrating — WLS p-value':30s} {m_p_mean:>10.4f}  {migr_p:>10.4f}")
# print(f"  {'Migrating — MK p-value':30s} {m_mk_p_mean:>10.4f}  {m_mk_p_median:>10.4f}")

WEIGHTED LINEAR REGRESSION ON MEDIAN

Spring
  Slope       : +0.1875 km/year  (SE = 0.0848)
  Intercept   : 35.6083
  R²          : 0.5499
  t-statistic : 2.2106
  p-value     : 0.0916  not significant

Autumn
  Slope       : -0.3072 km/year  (SE = 0.1238)
  Intercept   : 36.7859
  R²          : 0.6060
  t-statistic : -2.4805
  p-value     : 0.0682  not significant

Slope difference (spring − autumn): +0.4947 km/year
  SE of difference : 0.1501
  t-statistic      : 3.2956
  p-value          : 0.0109  *** significant

MANN-KENDALL EXACT TEST ON MEDIAN

spring
  S statistic : 11.0
  Kendall tau : +0.7333  (upward trend)
  p-value     : 0.0556  not significant

autumn
  S statistic : -9.0
  Kendall tau : -0.6000  (downward trend)
  p-value     : 0.1361  not significant


interpretation: failed the robustness check

the median for migrants being flat while the mean declines sharply suggests there are high-distance  observations being pulled down over time while other migrating birds haven't changed behavior, but extreme observations have shifted. Any test on the mean will capture that extreme effect, not necessarily a population-wide shift. Using the median gives it a more robust picture.

we were testing to see if most migrating birds are genuinely shifting closer to cities over time (a real population-level behavioural shift). Failing a robustness check could mean that i have fewer observations of migrating birds that get affected by nightlights. FIx this by obtaining more samples!!!
